# 融合模型（Regression）：SBERT (標題) + MLP (結構化特徵) → 預測 view_count

## 架構說明

```
標題 (text)
    └─► SBERT encoder ──► title_embedding (384維)
                                              \
                                               ► Fusion MLP ──► view_count (1個值, log scale)
                                              /
結構化特徵 (24維)
    └─► MLP backbone ──► feature_embedding (64維)
```

**輸出**：單一連續值（log1p(view_count)）  
**損失函數**：MSELoss  
**評估指標**：MAE、RMSE（log scale），以及還原後的 MAPE


In [ ]:
!pip install torch sentence-transformers numpy isodate scikit-learn -q

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from sentence_transformers import SentenceTransformer
from collections import OrderedDict

print(f"PyTorch version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
!rm -rf DL_Data_New
!git clone https://github.com/yitingCurry/DL_Data_New.git

## 3. 載入原始模型權重

In [ ]:
# ── 請修改成你的 .pt 檔路徑 ──
MLP_REG_MODEL_PATH      = "DL_Data_New/best_mlp_reg_model_state_dict.pt"
SBERT_REG_HEAD_PATH     = "DL_Data_New/regression_head.pt"
# 若仍需讀取 SBERT encoder name，請提供含有 sbert_model 欄位的 checkpoint
# 如果你有獨立的 sbert_ckpt（含 feature_mean/std 等），也請一併提供
SBERT_META_PATH         = "DL_Data_New/title_SBERT_classification.pt"   # 用來讀 sbert_model name / feature stats

# 載入 SBERT meta（只取 sbert_model name、embedding_dim、feature_mean/std）
sbert_ckpt = torch.load(SBERT_META_PATH, map_location='cpu', weights_only=False)
print("SBERT meta keys:", list(sbert_ckpt.keys()))
print(f"  sbert_model  : {sbert_ckpt['sbert_model']}")
print(f"  embedding_dim: {sbert_ckpt['embedding_dim']}")

# 載入 MLP regression state dict
mlp_reg_sd = torch.load(MLP_REG_MODEL_PATH, map_location='cpu', weights_only=False)
MLP_INPUT_DIM = mlp_reg_sd['mlp.0.weight'].shape[1]   # 24
print(f"\nMLP reg input_dim: {MLP_INPUT_DIM}")
print("MLP reg layer shapes:")
for k, v in mlp_reg_sd.items():
    if 'weight' in k and 'running' not in k and 'num_batches' not in k:
        print(f"  {k}: {v.shape}")

# 載入 regression head（Linear 384→1）
reg_head_sd = torch.load(SBERT_REG_HEAD_PATH, map_location='cpu', weights_only=False)
print(f"\nRegression head weight: {reg_head_sd['weight'].shape}")

## 4. 定義各子網路

In [ ]:
# ── MLP Backbone（Regression 版，24→64 中間表示）──
#
# 原始 best_mlp_reg_model_state_dict.pt 架構：
#   Linear(24→256)  → BN → ReLU → Dropout   (mlp.0~3)
#   Linear(256→256) → BN → ReLU → Dropout   (mlp.4~7)
#   Linear(256→128) → BN → ReLU → Dropout   (mlp.8~11)
#   Linear(128→128) → BN → ReLU → Dropout   (mlp.12~15)
#   Linear(128→64)  → BN → ReLU             (mlp.16~18)  ← backbone 輸出
#   Linear(64→1)                             (mlp.19)     ← 原 regression head，融合後捨棄
#
class MLPBackboneReg(nn.Module):
    def __init__(self, input_dim: int = 24, dropout: float = 0.3):
        super().__init__()
        self.mlp = nn.Sequential(
            # Block 1  (index 0-3)
            nn.Linear(input_dim, 256),  # mlp.0
            nn.BatchNorm1d(256),         # mlp.1
            nn.ReLU(),                   # mlp.2
            nn.Dropout(dropout),         # mlp.3
            # Block 2  (index 4-7)
            nn.Linear(256, 256),         # mlp.4
            nn.BatchNorm1d(256),         # mlp.5
            nn.ReLU(),                   # mlp.6
            nn.Dropout(dropout),         # mlp.7
            # Block 3  (index 8-11)
            nn.Linear(256, 128),         # mlp.8
            nn.BatchNorm1d(128),         # mlp.9
            nn.ReLU(),                   # mlp.10
            nn.Dropout(dropout),         # mlp.11
            # Block 4  (index 12-15)
            nn.Linear(128, 128),         # mlp.12
            nn.BatchNorm1d(128),         # mlp.13
            nn.ReLU(),                   # mlp.14
            nn.Dropout(dropout),         # mlp.15
            # Block 5  (index 16-18) ── backbone 輸出
            nn.Linear(128, 64),          # mlp.16
            nn.BatchNorm1d(64),          # mlp.17
            nn.ReLU(),                   # mlp.18
            # mlp.19 = Linear(64→1) 是原 regression head，不放在這裡
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.mlp(x)   # (B, 64)


# ── Fusion Model（Regression 版）──
class FusedModelReg(nn.Module):
    """
    輸入：
      - title_embedding : (B, 384)  — 由 SBERT 編碼的標題向量
      - struct_features : (B, 24)   — 結構化特徵

    架構：
      title_embedding (384) ──┐
                               ├─ concat (448) → Linear(448→256) → BN → ReLU → Dropout
      MLP backbone    (64)  ──┘               → Linear(256→128) → BN → ReLU → Dropout
                                              → Linear(128→1)   → 預測 log1p(view_count)
    """
    def __init__(
        self,
        sbert_dim:        int   = 384,
        mlp_backbone_dim: int   = 64,
        mlp_input_dim:    int   = 24,
        dropout:          float = 0.3,
    ):
        super().__init__()
        self.mlp_backbone = MLPBackboneReg(input_dim=mlp_input_dim, dropout=dropout)
        fused_dim = sbert_dim + mlp_backbone_dim   # 384 + 64 = 448

        self.fusion_head = nn.Sequential(
            nn.Linear(fused_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),   # 輸出 1 個值：log1p(view_count)
        )

    def forward(
        self,
        title_embedding: torch.Tensor,   # (B, 384)
        struct_features: torch.Tensor,   # (B, 24)
    ) -> torch.Tensor:
        feat  = self.mlp_backbone(struct_features)         # (B, 64)
        fused = torch.cat([title_embedding, feat], dim=1)  # (B, 448)
        out   = self.fusion_head(fused)                    # (B, 1)
        return out.squeeze(-1)                             # (B,)

print("✅ 網路定義完成")

## 5. 載入預訓練權重到融合模型

In [ ]:
def load_fused_reg_model(
    mlp_reg_ckpt_path: str,
    sbert_reg_head_path: str,
    mlp_input_dim: int,
    device,
) -> FusedModelReg:

    mlp_reg_sd  = torch.load(mlp_reg_ckpt_path,    map_location='cpu', weights_only=False)
    reg_head_sd = torch.load(sbert_reg_head_path,   map_location='cpu', weights_only=False)

    model = FusedModelReg(
        sbert_dim=384,
        mlp_backbone_dim=64,
        mlp_input_dim=mlp_input_dim,
    )

    # ── 1. 載入 MLP backbone 權重（排除原始 regression head mlp.19）──
    backbone_sd = OrderedDict(
        (k, v) for k, v in mlp_reg_sd.items()
        if not k.startswith('mlp.19')
    )
    missing, unexpected = model.mlp_backbone.load_state_dict(backbone_sd, strict=False)
    print("MLP backbone 載入完成")
    if missing:     print(f"  Missing    : {missing}")
    if unexpected:  print(f"  Unexpected : {unexpected}")

    # ── 2. fusion_head 初始化說明 ──
    # fusion_head 是全新的層，需要 fine-tune。
    # 如果你有預訓練好的 fusion_head 權重，可在此載入。
    print("Fusion head: 隨機初始化（需在資料上 fine-tune）")

    # ── 3. 選用：將 SBERT regression head 的權重作為 fusion_head 最後一層的初始化參考 ──
    # reg_head_sd 是 Linear(384→1)，而 fusion_head 最後一層是 Linear(128→1)
    # 維度不符，無法直接載入，但可以初始化 bias
    with torch.no_grad():
        model.fusion_head[-1].bias.fill_(reg_head_sd['bias'].item())
    print(f"Fusion head 最終層 bias 初始化為: {reg_head_sd['bias'].item():.4f}")

    model.to(device)
    return model


fused_model = load_fused_reg_model(
    MLP_REG_MODEL_PATH,
    SBERT_REG_HEAD_PATH,
    mlp_input_dim=MLP_INPUT_DIM,
    device=device,
)
print(f"\n模型參數量: {sum(p.numel() for p in fused_model.parameters()):,}")
print(fused_model)

## 6. 載入 SBERT encoder

In [ ]:
sbert_encoder = SentenceTransformer(sbert_ckpt['sbert_model'], device=str(device))

feature_mean = torch.tensor(sbert_ckpt['feature_mean'], dtype=torch.float32)
feature_std  = torch.tensor(sbert_ckpt['feature_std'],  dtype=torch.float32)

print(f"SBERT encoder 載入完成: {sbert_ckpt['sbert_model']}")

## 7. 資料預處理

In [ ]:
import re
import json
import pandas as pd
import isodate

def parse_iso8601_duration(duration_str):
    if not isinstance(duration_str, str):
        return 0
    try:
        return isodate.parse_duration(duration_str).total_seconds()
    except Exception:
        return 0

def parse_published_at(published_at_str):
    try:
        dt = pd.to_datetime(published_at_str)
        return {'pub_hour': dt.hour, 'pub_weekday': dt.dayofweek}
    except Exception:
        return {'pub_hour': 0, 'pub_weekday': 0}

In [ ]:
DATA_PATH = 'DL_Data_New/tw_youtube_videos.jsonl'

def preprocess_data(data_path):
    data = []
    with open(data_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    df = pd.DataFrame(data)
    df_encoded = df.copy()

    # 1. 移除 category_id == 10
    if 'category_id' in df_encoded.columns:
        df_encoded['category_id'] = pd.to_numeric(df_encoded['category_id'], errors='coerce')
        df_encoded = df_encoded[df_encoded['category_id'] != 10].copy()

    # 2. One-hot encoding for category_id
    df_encoded['category_id'] = df_encoded['category_id'].astype(str)
    df_encoded = pd.get_dummies(df_encoded, columns=['category_id'], prefix='cat')

    # 3. 數值特徵轉換
    if 'duration_iso8601' in df_encoded.columns:
        df_encoded['video_duration_sec'] = df_encoded['duration_iso8601'].apply(parse_iso8601_duration)

    if 'published_at' in df_encoded.columns:
        pub_info = df_encoded['published_at'].apply(lambda x: pd.Series(parse_published_at(str(x))))
        df_encoded = pd.concat([df_encoded, pub_info], axis=1)
        df_encoded['pub_hour_sin']     = np.sin(2 * np.pi * df_encoded['pub_hour']    / 24.0)
        df_encoded['pub_hour_cos']     = np.cos(2 * np.pi * df_encoded['pub_hour']    / 24.0)
        df_encoded['pub_weekday_sin']  = np.sin(2 * np.pi * df_encoded['pub_weekday'] /  7.0)
        df_encoded['pub_weekday_cos']  = np.cos(2 * np.pi * df_encoded['pub_weekday'] /  7.0)

    numeric_cols = ['subscriber_count', 'description_length', 'channel_view_count',
                    'channel_video_count', 'tags_count', 'like_count', 'comment_count', 'favorite_count']
    for col in numeric_cols:
        if col in df_encoded.columns:
            df_encoded[col] = pd.to_numeric(df_encoded[col], errors='coerce').fillna(0)

    # Log scale 壓縮
    for col in ['subscriber_count', 'video_duration_sec', 'description_length',
                'channel_view_count', 'channel_video_count']:
        if col in df_encoded.columns:
            df_encoded[col] = np.log1p(df_encoded[col])

    # ── Regression target：log1p(view_count) ──
    df_encoded['view_count'] = pd.to_numeric(df_encoded['view_count'], errors='coerce').fillna(0)
    df_encoded['log_view_count'] = np.log1p(df_encoded['view_count'].astype(float))

    # 特徵欄位
    all_cat_cols = [col for col in df_encoded.columns if col.startswith('cat_')]
    CAT_COLS = [col for col in all_cat_cols if re.match(r'cat_\d+$', col)]
    NUM_COLS = [
        'subscriber_count', 'video_duration_sec', 'tags_count', 'description_length',
        'pub_hour_sin', 'pub_hour_cos', 'pub_weekday_sin', 'pub_weekday_cos',
        'channel_view_count', 'channel_video_count',
    ]
    FEATURE_COLS = CAT_COLS + NUM_COLS

    for col in FEATURE_COLS:
        if col not in df_encoded.columns:
            df_encoded[col] = 0.0

    video_ids = df_encoded['video_id'].tolist()
    X         = df_encoded[FEATURE_COLS].values.astype(np.float32)
    y         = df_encoded['log_view_count'].values.astype(np.float32)   # 連續值
    titles    = df_encoded['title'].tolist()

    return titles, X, y, FEATURE_COLS, video_ids

all_titles, all_features, all_labels, FEATURE_COLS, all_video_ids = preprocess_data(DATA_PATH)
print(f"Total samples  : {len(all_titles)}")
print(f"Feature dim    : {all_features.shape[1]}")
print(f"Target (sample): mean={all_labels.mean():.3f}, std={all_labels.std():.3f}")
MLP_INPUT_DIM = all_features.shape[1]
print(f"MLP_INPUT_DIM  : {MLP_INPUT_DIM}")

## 8. 數據分割與 SBERT 嵌入

In [ ]:
from sklearn.preprocessing import StandardScaler

with open('DL_Data_New/data_split.json', 'r') as f:
    data_split_ids = json.load(f)

train_video_ids = data_split_ids['train']
val_video_ids   = data_split_ids['val']
test_video_ids  = data_split_ids['test']

temp_df = pd.DataFrame({
    'video_id': all_video_ids,
    'title':    all_titles,
    'features': list(all_features),
    'labels':   all_labels,
})

train_df = temp_df[temp_df['video_id'].isin(train_video_ids)]
val_df   = temp_df[temp_df['video_id'].isin(val_video_ids)]
test_df  = temp_df[temp_df['video_id'].isin(test_video_ids)]

train_titles   = train_df['title'].tolist()
train_features = np.array(train_df['features'].tolist())
train_labels   = train_df['labels'].values.astype(np.float32)

val_titles   = val_df['title'].tolist()
val_features = np.array(val_df['features'].tolist())
val_labels   = val_df['labels'].values.astype(np.float32)

test_titles   = test_df['title'].tolist()
test_features = np.array(test_df['features'].tolist())
test_labels   = test_df['labels'].values.astype(np.float32)

print(f"Train: {len(train_titles)}, Val: {len(val_titles)}, Test: {len(test_titles)}")

# Feature Scaling
scaler = StandardScaler()
train_features = scaler.fit_transform(train_features)
val_features   = scaler.transform(val_features)
test_features  = scaler.transform(test_features)

# SBERT embeddings
print("Generating SBERT embeddings for train...")
train_title_embeddings = sbert_encoder.encode(train_titles, convert_to_tensor=True, device=str(device)).cpu().numpy()
print("Generating SBERT embeddings for val...")
val_title_embeddings   = sbert_encoder.encode(val_titles,   convert_to_tensor=True, device=str(device)).cpu().numpy()
print("Generating SBERT embeddings for test...")
test_title_embeddings  = sbert_encoder.encode(test_titles,  convert_to_tensor=True, device=str(device)).cpu().numpy()
print("Done.")

## 9. Dataset & DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader

class TitleFeatureDatasetReg(Dataset):
    """
    title_embeddings : (N, 384)
    struct_features  : (N, 24)
    labels           : (N,) — log1p(view_count)
    """
    def __init__(self, title_embeddings, struct_features, labels):
        self.title_emb = torch.tensor(title_embeddings, dtype=torch.float32)
        self.feats     = torch.tensor(struct_features,  dtype=torch.float32)
        self.labels    = torch.tensor(labels,           dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.title_emb[idx], self.feats[idx], self.labels[idx]

BATCH_SIZE = 32

train_ds = TitleFeatureDatasetReg(train_title_embeddings, train_features, train_labels)
val_ds   = TitleFeatureDatasetReg(val_title_embeddings,   val_features,   val_labels)
test_ds  = TitleFeatureDatasetReg(test_title_embeddings,  test_features,  test_labels)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

## 10. 訓練函式

In [ ]:
def train_epoch_reg(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, total = 0.0, 0
    all_preds, all_labels = [], []

    for title_emb, feats, labels in loader:
        title_emb, feats, labels = title_emb.to(device), feats.to(device), labels.to(device)
        optimizer.zero_grad()
        preds = model(title_emb, feats)   # (B,)
        loss  = criterion(preds, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        total      += len(labels)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    mae  = np.mean(np.abs(np.array(all_preds) - np.array(all_labels)))
    rmse = np.sqrt(np.mean((np.array(all_preds) - np.array(all_labels)) ** 2))
    return total_loss / total, mae, rmse


@torch.no_grad()
def eval_epoch_reg(model, loader, criterion, device):
    model.eval()
    total_loss, total = 0.0, 0
    all_preds, all_labels = [], []

    for title_emb, feats, labels in loader:
        title_emb, feats, labels = title_emb.to(device), feats.to(device), labels.to(device)
        preds = model(title_emb, feats)
        loss  = criterion(preds, labels)
        total_loss += loss.item() * len(labels)
        total      += len(labels)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    mae  = np.mean(np.abs(all_preds - all_labels))
    rmse = np.sqrt(np.mean((all_preds - all_labels) ** 2))

    # MAPE 還原到原始空間
    pred_orig  = np.expm1(np.clip(all_preds,  0, None))
    label_orig = np.expm1(np.clip(all_labels, 0, None))
    mape = np.mean(np.abs((pred_orig - label_orig) / (label_orig + 1))) * 100

    return total_loss / total, mae, rmse, mape

print("✅ 訓練函式定義完成")

## 11. Fine-tune Fusion Head

In [ ]:
FREEZE_BACKBONE = True   # True = 只訓練 fusion head（建議先從這裡開始）
EPOCHS     = 100
LR         = 1e-3

if FREEZE_BACKBONE:
    for param in fused_model.mlp_backbone.parameters():
        param.requires_grad = False
    print("MLP backbone 已凍結，只訓練 fusion head")
else:
    for param in fused_model.parameters():
        param.requires_grad = True
    print("全部參數解凍，端對端 fine-tune")

trainable = sum(p.numel() for p in fused_model.parameters() if p.requires_grad)
print(f"可訓練參數量: {trainable:,}")

optimizer  = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, fused_model.parameters()),
    lr=LR, weight_decay=1e-4,
)
criterion  = nn.MSELoss()
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
best_val_rmse = float('inf')

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_mae, tr_rmse             = train_epoch_reg(fused_model, train_dl, optimizer, criterion, device)
    va_loss, va_mae, va_rmse, va_mape    = eval_epoch_reg(fused_model,  val_dl,   criterion, device)
    scheduler.step()

    flag = ""
    if va_rmse < best_val_rmse:
        best_val_rmse = va_rmse
        torch.save(fused_model.state_dict(), "fused_reg_model_best.pt")
        flag = " ← best RMSE"

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d}/{EPOCHS} | "
              f"train loss={tr_loss:.4f} MAE={tr_mae:.4f} | "
              f"val loss={va_loss:.4f} MAE={va_mae:.4f} RMSE={va_rmse:.4f} MAPE={va_mape:.1f}%{flag}")

print(f"\n最佳 Val RMSE (log scale): {best_val_rmse:.4f}")

## 12. 儲存最終融合模型

In [ ]:
save_payload = {
    'version':                   1,
    'task':                      'regression',
    'sbert_model':               sbert_ckpt['sbert_model'],
    'embedding_dim':             sbert_ckpt['embedding_dim'],
    'mlp_input_dim':             MLP_INPUT_DIM,
    'feature_mean':              sbert_ckpt['feature_mean'],
    'feature_std':               sbert_ckpt['feature_std'],
    'fused_model_state_dict':    fused_model.state_dict(),
    'note':                      'output = log1p(view_count); apply expm1() to recover',
}

torch.save(save_payload, "fused_reg_model_final.pt")
print("✅ 融合模型已儲存至 fused_reg_model_final.pt")

## 13. 測試集評估

In [ ]:
import matplotlib.pyplot as plt

# 載入最佳模型
best_model = FusedModelReg(
    sbert_dim=384,
    mlp_backbone_dim=64,
    mlp_input_dim=MLP_INPUT_DIM,
)
best_model.load_state_dict(torch.load("fused_reg_model_best.pt", map_location=device))
best_model.to(device)

_, te_mae, te_rmse, te_mape = eval_epoch_reg(best_model, test_dl, criterion, device)
print(f"Test MAE  (log scale) : {te_mae:.4f}")
print(f"Test RMSE (log scale) : {te_rmse:.4f}")
print(f"Test MAPE (原始空間)  : {te_mape:.2f}%")

# 收集預測值 & 真值
all_preds, all_labels_t = [], []
best_model.eval()
with torch.no_grad():
    for title_emb, feats, labels in test_dl:
        title_emb, feats = title_emb.to(device), feats.to(device)
        preds = best_model(title_emb, feats).cpu().numpy()
        all_preds.extend(preds)
        all_labels_t.extend(labels.numpy())

all_preds    = np.array(all_preds)
all_labels_t = np.array(all_labels_t)

# 視覺化：log scale 預測 vs 真值
plt.figure(figsize=(7, 6))
plt.scatter(all_labels_t, all_preds, alpha=0.3, s=8)
mn, mx = min(all_labels_t.min(), all_preds.min()), max(all_labels_t.max(), all_preds.max())
plt.plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='perfect')
plt.xlabel("True log1p(view_count)")
plt.ylabel("Predicted log1p(view_count)")
plt.title(f"Test Set Prediction\nMAE={te_mae:.3f}  RMSE={te_rmse:.3f}  MAPE={te_mape:.1f}%")
plt.legend()
plt.tight_layout()
plt.show()

# 視覺化：還原到原始空間（log scale）
pred_orig  = np.expm1(np.clip(all_preds,    0, None))
label_orig = np.expm1(np.clip(all_labels_t, 0, None))
plt.figure(figsize=(7, 6))
plt.scatter(np.log10(label_orig + 1), np.log10(pred_orig + 1), alpha=0.3, s=8)
plt.xlabel("True log10(view_count + 1)")
plt.ylabel("Predicted log10(view_count + 1)")
plt.title("Test Set — Original Scale (log10 axis)")
plt.tight_layout()
plt.show()

## 14. 推論函式

In [ ]:
@torch.no_grad()
def predict_view_count(
    titles:          list,
    struct_features: np.ndarray,   # (N, 24)
    model:           FusedModelReg,
    encoder:         SentenceTransformer,
) -> dict:
    """
    回傳：
      log_pred   : log1p(view_count) 預測值
      pred_views : 還原後的 view_count 預測值（整數）
    """
    model.eval()
    title_emb = encoder.encode(titles, convert_to_tensor=True, device=str(device)).float()
    feats     = torch.tensor(struct_features, dtype=torch.float32).to(device)

    log_pred   = model(title_emb, feats).cpu().numpy()          # (N,)
    pred_views = np.expm1(np.clip(log_pred, 0, None)).astype(int)

    return {
        'log_pred':   log_pred,
        'pred_views': pred_views,
    }

print("✅ 推論函式定義完成")